In [3]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/

In [4]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from crewai import LLM
llm = LLM(
    model="gemini/gemini-flash-lite-latest",
    temperature=0.5
    )

In [5]:
article = """Why Every Student Should Learn AI Agents in 2026

AI agents are programs that can plan, use tools and complete multi-step tasks
on their own. Unlike a chatbot that only answers questions, an agent can search
the web, read files, write reports and even work with other agents as a team.

Companies are already using agents for customer support, resume screening and
bug triage, saving hundreds of hours every month. This is creating demand for
developers who understand how to design them.

The good news is that you don't need expensive tools to start. Frameworks like
CrewAI and free models like Gemini let students build real projects on Google
Colab. The key skills are clear prompt writing, breaking problems into steps,
and knowing when to let code make decisions instead of the AI.

Students who start now will have a strong advantage in placements, because
very few freshers can show working agent projects on their resume."""

In [6]:
from crewai import Agent, Task, Crew

async def run_crew(role, goal, instructions, text):
  agent = Agent(
      role = role,
      goal = goal,
      backstory = "You are an experienced " + role + ".",
      llm = llm,
      verbose = False
  )
  task = Task(
      description = instructions + "\n\nContent:\n"+ text,
      expected_output = "The finished piece, ready to post.",
      agent = agent
  )

  crew = Crew(
      agents = [agent],
      tasks = [task],
      verbose = False
  )
  result = await crew.kickoff_async()
  return result.raw

In [7]:
from crewai.flow.flow import Flow, start, listen, and_

class RepurposeFlow(Flow):

    @start()
    def load_article(self):
        print("📄 Article loaded:", len(self.state["article"]), "characters")

    @listen(load_article)
    def extract_points(self):
        points = llm.call(
            "List the 5 most important points from this article as short bullets.\n\n"
            + self.state["article"]
        )
        self.state["points"] = points.strip()
        print("Key points:\n" + self.state["points"])

    @listen(extract_points)
    async def write_linkedin(self):
        self.state["linkedin"] = await run_crew(
            role="LinkedIn Content Writer",
            goal="Write engaging LinkedIn posts",
            instructions="Write a LinkedIn post of about 150 words from these points. "
                         "Start with a strong one-line hook. End with a question.",
            text=self.state["points"],
        )

    @listen(extract_points)
    async def write_twitter(self):
        self.state["twitter"] = await run_crew(
            role="Twitter Content Writer",
            goal="Write punchy tweet threads",
            instructions="Write a thread of exactly 6 tweets from these points. "
                         "Number them 1/6 to 6/6. Each tweet under 280 characters.",
            text=self.state["points"],
        )

    @listen(extract_points)
    async def write_newsletter(self):
        self.state["newsletter"] = await run_crew(
            role="Newsletter Editor",
            goal="Write friendly newsletter sections",
            instructions="Write a newsletter section: a 2-line friendly intro, "
                         "then 3 bullet points, then a one-line sign-off.",
            text=self.state["points"],
        )

    @listen(and_(write_linkedin, write_twitter, write_newsletter))
    def publish_pack(self):
        for name in ["linkedin", "twitter", "newsletter"]:
            with open(name + ".md", "w") as f:
                f.write(self.state[name])
        print("Saved: linkedin.md, twitter.md, newsletter.md")

In [8]:
flow = RepurposeFlow()
await flow.kickoff_async(inputs={"article": article})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: RepurposeFlow                                                                                            │
│  ID: 2c3ff535-04aa-4dec-b748-ecf4a97d0d27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: RepurposeFlow                                                                                            │
│  ID: 2c3ff535-04aa-4dec-b748-ecf4a97d0d27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📄 Article loaded: 930 characters


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_article                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_article                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: write_twitter                                                                                          │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Saved: linkedin.md, twitter.md, newsletter.md


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_pack                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_pack                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: RepurposeFlow                                                                                            │
│  ID: 2c3ff535-04aa-4dec-b748-ecf4a97d0d27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
from IPython.display import Markdown, display

for name in ["linkedin", "twitter", "newsletter"]:
    print("=" * 20, name.upper(), "=" * 20)
    with open(name + ".md") as f:
        display(Markdown(f.read()))

==================== LINKEDIN ====================


AI agents are no longer science fiction—they are completely reshaping the modern workplace. 

If you're a student or early-career professional, mastering them is your ultimate career cheat code. Here is what you need to know:

* **Autonomous Power:** Agents can now independently plan, use tools, and execute complex, multi-step tasks.
* **Rapid Adoption:** Companies are leveraging them for everything from customer support to bug triage, creating massive demand for agentic skills.
* **Zero-Cost Learning:** You don't need a massive budget. Students can build real projects for free using CrewAI, Gemini, and Google Colab.
* **The Core Skills:** Success comes down to three things: writing clear prompts, breaking down massive problems, and knowing when to keep a human in the loop.
* **Instant Advantage:** Building working projects *today* gives you a portfolio that sets you apart from 99% of applicants tomorrow.

Which AI framework are you planning to experiment with first?

==================== TWITTER ====================


1/6
AI is shifting from simple chatbots to autonomous agents. These systems can independently plan, use tools, and execute complex, multi-step tasks without hand-holding. A massive shift is underway. 🧵👇

2/6
Industry adoption is exploding. Companies are deploying AI agents to automate time-draining tasks like customer support and bug triage. This practical shift is creating massive demand for agent-building skills. 🚀

3/6
The best part? You don't need a massive budget to start. Students can build real, working projects for free using powerful, accessible tools like CrewAI, Gemini, and Google Colab. No excuses. 💻✨

4/6
What core skills do you actually need? Success isn't about complex coding—it's about mastering clear prompt writing, breaking down large problems, and knowing when to let AI vs. humans make decisions. 🧠

5/6
Learning this right now gives students an unfair career advantage. You can land jobs simply by showcasing rare, working AI projects in your portfolio instead of just listing theoretical skills on a resume. 📈

6/6
AI agents are the future of work, and the barrier to entry has never been lower. Start building today, master the core logic, and put yourself ahead of 99% of your peers. 

What agent will you build first? Let me know below! 👇

==================== NEWSLETTER ====================


Hey friends! Ready to dive into the exciting world of AI agents and see what everyone's buzzing about? Let's jump right into the big takeaways you need to know today!

* **AI agents are stepping up:** These clever systems can now independently plan, use tools, and knock out complex, multi-step tasks all on their own.
* **Building blocks are totally free:** Students can dive right in and build real projects using accessible tools like CrewAI, Gemini, and Google Colab without spending a dime.
* **Your new career superpower:** Getting ahead of the curve now gives you a massive competitive edge by showing off rare, working projects to future employers.

Catch you in the next issue, and keep building!